# Amazon Business Entity Resolution — Kaggle run (T4x2, 3h budget)
Public data: `/kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset` (input-only). No external business lookup / geocoding / APIs. Pretrained encoder only (`minishlab/potion-multilingual-128M`, MIT).

In [ ]:
!git clone https://github.com/BhanuPrakash-004/Amazon-ML-Challenge.git 2>&1 | tail -1
%cd Amazon-ML-Challenge/code/business_entity_resolution
!pip install -r requirements.txt 2>&1 | tail -2
!nvidia-smi --query-gpu=index,name,memory.total --format=csv 2>&1 | head -5
!ls /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/train

In [ ]:
# Phase 0 (mandatory): Blocking A/B/C/D experiment — pick smallest set with ~>=99% recall
!python -m src.main --experiment --K 22 2>&1 | tail -15

In [ ]:
# Full pipeline: train (LightGBM + F0.5 thresholds) then test inference + validator.
# Budget <3h with 5min safety. GPU0=S2 lane, GPU1=S3 lane, CPU-FAISS fallback.
!python -m src.main --mode full --K 22 --batch-size 150000 --workers 4 --gpu-ids 0,1 --train-sample 250000 2>&1 | tail -30

In [ ]:
!python ../../../student_resource/utils/validate_submission.py --matching ../../output/matching_results.tsv --candidate ../../output/candidate_pairs.tsv --test-dir /kaggle/input/datasets/satwiksps/amazon-ml-challenge-2026/dataset/test
!head -3 ../../output/matching_results.tsv && wc -l ../../output/matching_results.tsv